## A/B testing

In [13]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/checking-logs.sqlite")
schema = pd.io.sql.read_sql("PRAGMA table_info(test);", conn)
schema

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


## before\after first pageview for test

In [14]:
query_test = """
SELECT
    'before' as time,
    AVG((strftime('%s', t.first_commit_ts)-d.deadlines) / 3600) as avg_diff
FROM
    test t
JOIN
    deadlines d ON t.labname = d.labs
WHERE
    t.labname != 'project1'
    AND t.first_commit_ts < t.first_view_ts
    AND t.uid IN (
        SELECT uid
        FROM test
        WHERE first_commit_ts >= first_view_ts
    )

UNION ALL

SELECT
    'after' as time,
    AVG((strftime('%s', t.first_commit_ts)-d.deadlines) / 3600) as avg_diff
FROM
    test t
JOIN
    deadlines d ON t.labname = d.labs
WHERE
    t.labname != 'project1'
    AND t.first_commit_ts >= t.first_view_ts
    AND t.uid IN (
        SELECT uid
        FROM test
        WHERE first_commit_ts < first_view_ts
    )
"""

test_results = pd.io.sql.read_sql(query_test, conn)
test_results

,time,avg_diff
0,before,-60.5625
1,after,-93.8500


## before after for control

In [15]:
query_control = """
SELECT
    'before' as time,
    AVG((d.deadlines - strftime('%s', t.first_commit_ts)) / 3600) as avg_diff
FROM
    control t
JOIN
    deadlines d ON t.labname = d.labs
WHERE
    t.labname != 'project1'
    AND t.first_commit_ts < t.first_view_ts
    AND t.uid IN (
        SELECT uid
        FROM control
        WHERE first_commit_ts >= first_view_ts
    )

UNION ALL

SELECT
    'after' as time,
    AVG((d.deadlines - strftime('%s', t.first_commit_ts)) / 3600) as avg_diff
FROM
    control t
JOIN
    deadlines d ON t.labname = d.labs
WHERE
    t.labname != 'project1'
    AND t.first_commit_ts >= t.first_view_ts
    AND t.uid IN (
        SELECT uid
        FROM control
        WHERE first_commit_ts < first_view_ts
    )
"""

control_results = pd.io.sql.read_sql(query_control, conn)
control_results

,time,avg_diff
0,before,99.464286
1,after,117.636364


## "Did the hypothesis turn out to be true, and does the page affect students' behavior?"

### Да но лишь с небольшой разницей. Посколько студенты которые не видели страниц стали решать задачи тоже быстрее, возможно просто сложность стала ниже, или студенты лучше адаптировались к задачам. разниа у тестовой группы - 8 часов, а у группы контроль - 6, между до и после pageviews. следовательно в среднем студенты тестовой группы имеют лучшие показатели на 2 часа быстрее.

In [16]:
conn.close()